# z002 - Modelo base LightGBM (versión A: solo BAJA+2)

**Objetivo:** tener un primer modelo para:
1. Ver cuánta **ganancia** da en un mes que no vio (validación).
2. Entender las **métricas** del resumen: AUC, KS, lift y captura.
3. Ver qué **variables importan** (y si `ccajas_depositos` es importante).

Es el **punto de referencia**: todo lo que agreguemos después (lags, Optuna, etc.) se compara contra este número.

**Esquema:** entrenar con `MESES_TRAIN`, validar con 202106 (2 meses de distancia, igual que 202106 → 202108).

In [ ]:
# --- 0. Instalar LightGBM (correr UNA sola vez; después podés comentar esta línea) ---
# El % hace que se instale en el mismo entorno (.venv) que usa este notebook
%pip install lightgbm

In [ ]:
# --- 1. Librerías y parámetros generales ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
# (AUC y KS se calculan a mano en la celda 6: Windows bloquea sklearn.metrics)

# Fórmula de ganancia (de z601)
GANANCIA_ACIERTO = 1_072_500   # estimular a un BAJA+2
COSTO_ESTIMULO   = 27_500      # estimular a cualquier otro
UMBRAL = COSTO_ESTIMULO / (GANANCIA_ACIERTO + COSTO_ESTIMULO)   # = 0.025

# Meses: probá primero [202104] y después [202103, 202104]
MESES_TRAIN = [202104]
MES_VALID   = 202106

SEMILLA = 501419   # tu semilla de z301

print("Umbral de equilibrio:", UMBRAL)

In [ ]:
# --- 2. Cargar datos y armar el target binario ---
df = pd.read_csv("../competencia_01.csv")

# 1 si es BAJA+2, 0 en cualquier otro caso
df["clase_binaria"] = (df["clase_ternaria"] == "BAJA+2").astype(int)

# Separamos entrenamiento y validación por mes
train = df[df["foto_mes"].isin(MESES_TRAIN)]
valid = df[df["foto_mes"] == MES_VALID]

# Columnas que NO son variables para el modelo:
# - el ID del cliente (es un número de identificación, no dice nada del cliente)
# - el mes
# - el target (¡sería hacer trampa!)
no_usar = ["numero_de_cliente", "foto_mes", "clase_ternaria", "clase_binaria"]
variables = [c for c in df.columns if c not in no_usar]

X_train, y_train = train[variables], train["clase_binaria"]
X_valid, y_valid = valid[variables], valid["clase_binaria"]

print("Entrenamiento:", X_train.shape, "| BAJA+2:", y_train.sum())
print("Validación:   ", X_valid.shape, "| BAJA+2:", y_valid.sum())

In [ ]:
# --- 3. Entrenar LightGBM con hiperparámetros razonables (SIN tunear) ---
params = {
    "objective": "binary",       # problema de 2 clases: devuelve probabilidad de BAJA+2
    "learning_rate": 0.05,       # cuánto aporta cada árbol
    "num_leaves": 31,            # hojas máximas por árbol (valor por defecto)
    "min_data_in_leaf": 100,     # mínimo de clientes por hoja (frena el sobreajuste)
    "feature_fraction": 0.8,     # cada árbol usa el 80% de las variables, al azar
    "seed": SEMILLA,
    "verbose": -1,               # que no imprima mensajes de más
}

datos_lgb = lgb.Dataset(X_train, label=y_train)

# num_boost_round = cantidad de árboles
modelo = lgb.train(params, datos_lgb, num_boost_round=300)

# Probabilidad de BAJA+2 para cada cliente de validación
prob_valid = modelo.predict(X_valid)
print("Listo. Probabilidad promedio:", prob_valid.mean().round(4))

In [ ]:
# --- 4. Ganancia con corte en probabilidad 0.025 ---
# Ganancia de cada cliente SI lo estimulamos: +acierto si es BAJA+2, -costo si no
ganancia_cliente = np.where(y_valid == 1, GANANCIA_ACIERTO, -COSTO_ESTIMULO)

# Estimulamos a los que superan el umbral
estimulados = prob_valid >= UMBRAL

ganancia_umbral = ganancia_cliente[estimulados].sum()
print("Clientes estimulados:", estimulados.sum())
print("Ganancia:", round(ganancia_umbral / 1e6, 1), "millones")

In [ ]:
# --- 5. Ganancia acumulada: ¿cuántos clientes conviene estimular? ---
# Ordenamos de mayor a menor probabilidad
orden = np.argsort(prob_valid)[::-1]

# Vamos sumando la ganancia cliente por cliente, en ese orden
ganancia_acum = np.cumsum(ganancia_cliente[orden])

# El máximo de la curva = el mejor corte
mejor_n = np.argmax(ganancia_acum) + 1
mejor_ganancia = ganancia_acum[mejor_n - 1]
prob_en_corte = prob_valid[orden][mejor_n - 1]

print("Mejor cantidad a estimular:", mejor_n)
print("Ganancia máxima:", round(mejor_ganancia / 1e6, 1), "millones")
print("Probabilidad en ese corte:", round(prob_en_corte, 4), "(compará con 0.025)")

# Gráfico de los primeros 20.000 clientes
limite = 20_000
plt.figure(figsize=(10, 4))
plt.plot(range(1, limite + 1), ganancia_acum[:limite] / 1e6)
plt.axvline(mejor_n, color="red", linestyle="--", label=f"Corte óptimo: {mejor_n}")
plt.axvline(estimulados.sum(), color="green", linestyle=":", label="Corte 0.025")
plt.axvspan(8_000, 15_000, color="gray", alpha=0.15, label="Rango de la consigna")
plt.xlabel("Cantidad de clientes estimulados (de mayor a menor probabilidad)")
plt.ylabel("Ganancia acumulada (millones)")
plt.legend()
plt.show()

In [ ]:
# --- 6. Métricas del resumen ---
# Ordenamos a los clientes de mayor a menor probabilidad
y_ord = y_valid.values[np.argsort(prob_valid)[::-1]]
n_bajas, n_resto = y_ord.sum(), len(y_ord) - y_ord.sum()

# Recorriendo esa lista, en cada punto calculamos:
# TPR = de todas las BAJA+2, qué proporción ya encontramos
# FPR = de todos los demás, qué proporción ya estimulamos de más
tpr = np.cumsum(y_ord) / n_bajas
fpr = np.cumsum(1 - y_ord) / n_resto

# AUC: área bajo la curva TPR vs FPR (0.5 = azar, 1 = perfecto)
auc = np.trapezoid(np.r_[0, tpr], np.r_[0, fpr])

# KS: la máxima distancia entre TPR y FPR
ks = (tpr - fpr).max()

# Lift y captura en el grupo estimulado (corte 0.025)
tasa_general  = y_valid.mean()                    # % de BAJA+2 en todo el mes
tasa_elegidos = y_valid[estimulados].mean()       # % de BAJA+2 entre los estimulados
lift = tasa_elegidos / tasa_general
captura = y_valid[estimulados].sum() / y_valid.sum()

print(f"AUC:     {auc:.3f}")
print(f"KS:      {ks:.3f}")
print(f"Tasa de BAJA+2 general:        {tasa_general:.2%}")
print(f"Tasa de BAJA+2 en estimulados: {tasa_elegidos:.2%}")
print(f"Lift:    {lift:.1f}  (veces más bajas que al azar)")
print(f"Captura: {captura:.1%}  (de todas las BAJA+2, cuántas encontramos)")

In [ ]:
# --- 7. Importancia de variables ---
# "gain" = cuánto mejoró el modelo gracias a los cortes en cada variable
importancia = pd.DataFrame({
    "variable": variables,
    "importancia": modelo.feature_importance(importance_type="gain"),
}).sort_values("importancia", ascending=False).reset_index(drop=True)

# Porcentaje sobre el total, más fácil de leer
importancia["pct"] = (importancia["importancia"] / importancia["importancia"].sum() * 100).round(2)

# ¿En qué puesto quedó la variable rota? (+1 porque el índice empieza en 0)
puesto = importancia.index[importancia["variable"] == "ccajas_depositos"][0] + 1
print("ccajas_depositos quedó en el puesto", puesto, "de", len(variables))

importancia.head(20)

## Conclusiones (completar después de correr)

| Prueba | MESES_TRAIN | Ganancia 0.025 | Ganancia máxima | Mejor N | AUC |
|---|---|---|---|---|---|
| Base | [202104] | ... | ... | ... | ... |
| Más meses | [202103, 202104] | ... | ... | ... | ... |

- ¿El corte óptimo cae cerca de 0.025?
- ¿Cae dentro del rango 8.000–15.000?
- Puesto de `ccajas_depositos`: ...
- Top 5 variables: ...